<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/08_Base_de_Conhecimento_e_Regras_de_Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 08 – Sistemas Baseados em Conhecimento & Regras de Produção SE-ENTÃO

## 1. Fundamentação Teórica
Implementação do Sistema Especialista com Base de Regras declarativas, Memória de Trabalho dinâmica, resolução de conflitos por prioridade estática e diagnóstico multivariável em tempo real.

In [1]:
import numpy as np

class ProductionRule:
    def __init__(self, rule_id, name, condition_fn, action_fact, priority=1, subsystem="GERAL"):
        self.rule_id = rule_id
        self.name = name
        self.condition_fn = condition_fn
        self.action_fact = action_fact
        self.priority = priority
        self.subsystem = subsystem

    def evaluate(self, working_memory):
        return self.condition_fn(working_memory)


class ExpertDiagnosticSystem:
    def __init__(self):
        self.rules = []
        self.working_memory = {}
        self.active_diagnostics = []
        self.fired_log = []

    def add_rule(self, rule):
        self.rules.append(rule)

    def set_telemetry(self, telemetry_dict):
        self.working_memory = dict(telemetry_dict)
        self.working_memory['P_PUMP_ON'] = self.working_memory.get('DRN_PMP_01', 0.0) > 5.0
        self.working_memory['P_PRESS_HIGH'] = self.working_memory.get('DRN_PT_01', 0.0) >= 4.5
        self.working_memory['P_PRESS_LOW'] = self.working_memory.get('DRN_PT_01', 0.0) <= 1.2
        self.working_memory['P_FLOW_LOW'] = self.working_memory.get('DRN_FT_01', 0.0) <= 0.4
        self.working_memory['P_FLOW_HIGH'] = self.working_memory.get('DRN_FT_01', 0.0) >= 3.0
        self.working_memory['P_DRN_EMPTY'] = self.working_memory.get('DRN_LT_01', 100.0) <= 3.0
        self.working_memory['P_VENTO_ALTO'] = self.working_memory.get('EST_WT_01', 0.0) >= 15.0
        self.working_memory['P_CORRENTE_ALTA'] = self.working_memory.get('DRN_ET_01_CORRENTE', 0.0) >= 55.0
        self.working_memory['P_TAXA_QUEDA_RAPIDA'] = self.working_memory.get('DRN_ET_01_DV_DT', 0.0) <= -0.2

    def run_inference_cycle(self):
        self.active_diagnostics.clear()
        self.fired_log.clear()

        conflict_set = []
        for rule in self.rules:
            if rule.evaluate(self.working_memory):
                conflict_set.append(rule)

        conflict_set.sort(key=lambda r: r.priority, reverse=True)

        for rule in conflict_set:
            fact_entry = {
                'rule_id': rule.rule_id,
                'name': rule.name,
                'subsystem': rule.subsystem,
                'priority': rule.priority,
                'diagnosis': rule.action_fact
            }
            self.active_diagnostics.append(fact_entry)
            self.fired_log.append(f"[DISPARO Prio {rule.priority}] {rule.rule_id}: {rule.name} -> {rule.action_fact}")

        return self.active_diagnostics


scada_expert = ExpertDiagnosticSystem()

scada_expert.add_rule(ProductionRule(
    "R1_OBSTR_BICOS", "Obstrução de Bicos Centrífugos",
    lambda wm: wm['P_PUMP_ON'] and wm['P_PRESS_HIGH'] and wm['P_FLOW_LOW'],
    "CRÍTICO: Obstrução hidráulica detectada! Sobrepressão com subvazão. Inibir bomba.",
    priority=3, subsystem="HIDRAULICA"
))

scada_expert.add_rule(ProductionRule(
    "R2_RUPTURA_LINHA", "Ruptura de Tubulação / Desconexão",
    lambda wm: wm['P_PUMP_ON'] and wm['P_PRESS_LOW'] and wm['P_FLOW_HIGH'],
    "EMERGÊNCIA: Ruptura de mangueira! Vazão excessiva com perda de pressão. Corte instantâneo da bomba.",
    priority=4, subsystem="HIDRAULICA"
))

scada_expert.add_rule(ProductionRule(
    "R3_CAVITACAO", "Risco de Cavitação da Bomba",
    lambda wm: wm['P_PUMP_ON'] and wm['P_DRN_EMPTY'],
    "ALERTA: Bomba acionada com reservatório vazio. Desligar para evitar queima a seco.",
    priority=3, subsystem="HIDRAULICA"
))

scada_expert.add_rule(ProductionRule(
    "R4_DESCARGA_BATERIA", "Descarga Crítica Acelerada de Bateria",
    lambda wm: wm['P_CORRENTE_ALTA'] and wm['P_TAXA_QUEDA_RAPIDA'],
    "EMERGÊNCIA: Curto ou degradação química da LiPo! Disparo de Failsafe AUTO-RTH.",
    priority=4, subsystem="ELETRICA"
))

scada_expert.add_rule(ProductionRule(
    "R5_DERIVA_VENTO", "Risco de Deriva Excessiva de Produto",
    lambda wm: wm['P_VENTO_ALTO'] and wm['P_PUMP_ON'],
    "ALERTA AGRONÔMICO: Vento acima de 15 km/h durante pulverização. Pausar aplicação.",
    priority=2, subsystem="METEOROLOGIA"
))

scada_expert.add_rule(ProductionRule(
    "R6_SENSOR_FROZEN", "Sensor de Pressão DRN_PT_01 Travado",
    lambda wm: (not wm['P_PUMP_ON']) and wm['P_PRESS_HIGH'],
    "MANUTENÇÃO: Sensor DRN_PT_01 indicando pressão alta com bomba desligada. Calibrar transmissor.",
    priority=1, subsystem="INSTRUMENTACAO"
))

print("=" * 80)
print("TESTE DO SISTEMA ESPECIALISTA DE DIAGNÓSTICO SCADA AGRODRONE")
print("=" * 80)

telemetria_cenario_A = {
    'DRN_PMP_01': 65.0,
    'DRN_PT_01': 5.4,
    'DRN_FT_01': 0.15,
    'DRN_LT_01': 14.0,
    'EST_WT_01': 18.2,
    'DRN_ET_01_CORRENTE': 32.0,
    'DRN_ET_01_DV_DT': -0.02
}

scada_expert.set_telemetry(telemetria_cenario_A)
diagnosticos_A = scada_expert.run_inference_cycle()

print("\n--- CENÁRIO A: Telemetria com Falha Múltipla Concorrente ---")
for log in scada_expert.fired_log:
    print(f" * {log}")

print("\nDIAGNÓSTICOS EMITIDOS NA IHM (ORDEM DE CRITICIDADE):")
for d in diagnosticos_A:
    print(f" [{d['subsystem']:<14}] (Prio {d['priority']}) -> {d['diagnosis']}")
print("=" * 80)


TESTE DO SISTEMA ESPECIALISTA DE DIAGNÓSTICO SCADA AGRODRONE

--- CENÁRIO A: Telemetria com Falha Múltipla Concorrente ---
 * [DISPARO Prio 3] R1_OBSTR_BICOS: Obstrução de Bicos Centrífugos -> CRÍTICO: Obstrução hidráulica detectada! Sobrepressão com subvazão. Inibir bomba.
 * [DISPARO Prio 2] R5_DERIVA_VENTO: Risco de Deriva Excessiva de Produto -> ALERTA AGRONÔMICO: Vento acima de 15 km/h durante pulverização. Pausar aplicação.

DIAGNÓSTICOS EMITIDOS NA IHM (ORDEM DE CRITICIDADE):
 [HIDRAULICA    ] (Prio 3) -> CRÍTICO: Obstrução hidráulica detectada! Sobrepressão com subvazão. Inibir bomba.
 [METEOROLOGIA  ] (Prio 2) -> ALERTA AGRONÔMICO: Vento acima de 15 km/h durante pulverização. Pausar aplicação.


### 8. Exercício Proposto com Resolução Computacional

**Enunciado:**
Em uma missão de pulverização noturna, a estação SCADA recebe a seguinte telemetria suspeita:
* Bomba de pulverização desligada: `DRN_PMP_01 = 0.0%` ($P_{PUMP\_ON} = 0$).
* Sensor de pressão piezorresistivo indicando: `DRN_PT_01 = 4.8 bar` ($P_{PRESS\_HIGH} = 1$).
* Sensor de fluxo eletromagnético indicando: `DRN_FT_01 = 0.0 L/min` ($P_{FLOW\_LOW} = 1$).
* Corrente do inversor da bomba: `I_bomba = 0.1 A` (Confirmando repouso elétrico do motor).

1. Qual falha física real está ocorrendo (problema hidráulico ou erro de instrumentação)?
2. Adicione uma nova regra na Base de Conhecimento especializada: `R7_FALHA_DISCREPANCIA_SENSOR_PRESSAO` com prioridade $\rho=2$.
3. Execute o motor de inferência em Python para processar essa telemetria e comprove que o sistema emite o diagnóstico de falha de calibração do sensor sem acionar alarmes espúrios de obstrução de bico.

**Dica Metodológica:**
A regra de obstrução exige $P_{PUMP\_ON} = 1$. Como a bomba está desligada e não há fluxo nem consumo elétrico, a leitura de $4.8\text{ bar}$ constitui uma contradição física (*Incoerência de Sensor*).

**Resolução Computacional:**

In [2]:
# Resolução do Exercício 08
regra_sensor_discrepante = ProductionRule(
    "R7_SENSOR_PRESSAO_DESCALIBRADO",
    "Incoerência no Transmissor de Pressão DRN_PT_01",
    lambda wm: (not wm['P_PUMP_ON']) and wm['P_PRESS_HIGH'] and (wm.get('I_bomba_A', 0.0) < 0.5),
    "ALERTA DE INSTRUMENTAÇÃO: Transmissor DRN_PT_01 reportando 4.8 bar com bomba desligada e sem corrente. Sensor descalibrado!",
    priority=2,
    subsystem="INSTRUMENTACAO"
)

scada_expert.add_rule(regra_sensor_discrepante)

telemetria_exercicio = {
    'DRN_PMP_01': 0.0,
    'DRN_PT_01': 4.8,
    'DRN_FT_01': 0.0,
    'DRN_LT_01': 50.0,
    'EST_WT_01': 8.0,
    'DRN_ET_01_CORRENTE': 15.0,
    'DRN_ET_01_DV_DT': -0.01,
    'I_bomba_A': 0.1
}

scada_expert.set_telemetry(telemetria_exercicio)
diag_ex = scada_expert.run_inference_cycle()

print("=" * 80)
print("RESULTADO DO EXERCÍCIO 08 - AUDITORIA DE INTEGRIDADE DE SENSORES")
print("=" * 80)
print(f"Total de regras disparadas: {len(diag_ex)}")
for d in diag_ex:
    print(f" Regra: [{d['rule_id']}] -> {d['name']}")
    print(f" Diagnóstico Gerado: {d['diagnosis']}")
print("-" * 80)
print("Conclusão: O sistema especialista isolou corretamente o erro no sensor sem falsos alarmes!")
print("=" * 80)


RESULTADO DO EXERCÍCIO 08 - AUDITORIA DE INTEGRIDADE DE SENSORES
Total de regras disparadas: 2
 Regra: [R7_SENSOR_PRESSAO_DESCALIBRADO] -> Incoerência no Transmissor de Pressão DRN_PT_01
 Diagnóstico Gerado: ALERTA DE INSTRUMENTAÇÃO: Transmissor DRN_PT_01 reportando 4.8 bar com bomba desligada e sem corrente. Sensor descalibrado!
 Regra: [R6_SENSOR_FROZEN] -> Sensor de Pressão DRN_PT_01 Travado
 Diagnóstico Gerado: MANUTENÇÃO: Sensor DRN_PT_01 indicando pressão alta com bomba desligada. Calibrar transmissor.
--------------------------------------------------------------------------------
Conclusão: O sistema especialista isolou corretamente o erro no sensor sem falsos alarmes!
